# ⚡ Artificial Neural Network (ANN) Energy Consumption Predictor

In [ ]:
# Imports
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
# Load Dataset
df = pd.read_csv("dataset/energy.csv")
print(f"Total dataset shape: {df.shape}")
df.head()

In [ ]:
# Dataset Overview
print("Dataset Columns:")
print(df.columns)

print("\nDataset Info:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# DATA PREPROCESSING & FEATURE ENGINEERING

df = df.dropna().copy()

# Datetime & cyclical features
df['date'] = pd.to_datetime(df['date'])
df['hour'] = df['date'].dt.hour
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['dayofweek'] = df['date'].dt.dayofweek

df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24.0)

# Environmental averages & interaction terms
temp_cols = [c for c in df.columns if c.startswith('T') and c not in ['T_out', 'Tdewpoint']]
rh_cols = [c for c in df.columns if c.startswith('RH_') and c != 'RH_out']

if temp_cols:
    df['T_indoor_avg'] = df[temp_cols].mean(axis=1)
if rh_cols:
    df['RH_indoor_avg'] = df[rh_cols].mean(axis=1)

if 'T1' in df.columns and 'RH_1' in df.columns:
    df['T1_RH1_interaction'] = df['T1'] * df['RH_1']

df = df.drop('date', axis=1)

# Features and target
y = df['Appliances'].values
X = df.drop('Appliances', axis=1)
feature_names = list(X.columns)

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing completed successfully!")
print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

In [ ]:
# ANN MODEL BUILDING & TRAINING

def build_ann_model(input_dim):
    model = Sequential([
        Dense(128, activation='relu', input_dim=input_dim),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.1),
        Dense(32, activation='relu'),
        Dense(1, activation='linear')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='mean_squared_error',
        metrics=['mean_absolute_error']
    )
    return model

model = build_ann_model(X_train_scaled.shape[1])
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1)
]

print("\nStarting ANN Model Training...")
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# Save model & scaler
os.makedirs("results", exist_ok=True)
os.chmod("results", 0o700)
model.save("results/energy_model.keras")
joblib.dump({"scaler": scaler, "feature_names": feature_names}, "results/scaler.pkl")
print("Model & scaler saved successfully to results/")

In [ ]:
# MODEL EVALUATION & VISUALIZATION

y_pred = model.predict(X_test_scaled).flatten()

mae = float(mean_absolute_error(y_test, y_pred))
rmse = float(math.sqrt(mean_squared_error(y_test, y_pred)))
r2 = float(r2_score(y_test, y_pred))
mape = float(np.mean(np.abs((y_test - y_pred) / np.maximum(y_test, 1))) * 100)

print("=" * 40)
print("MODEL EVALUATION METRICS")
print("=" * 40)
print(f"Mean Absolute Error (MAE)  : {mae:.2f} Wh")
print(f"Root Mean Squared Error    : {rmse:.2f} Wh")
print(f"R-Squared (R2 Score)       : {r2:.4f}")
print(f"Mean Abs Percentage Error  : {mape:.2f}%")
print("=" * 40)

# Plot 1: Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Train Loss (MSE)', color='#2b5c8f')
plt.plot(history.history['val_loss'], label='Val Loss (MSE)', color='#d9534f', linestyle='--')
plt.title('ANN Training & Validation Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot 2: Actual vs Predicted Scatter Plot
plt.figure(figsize=(9, 5))
plt.scatter(y_test[:300], y_pred[:300], alpha=0.6, color='#2b5c8f', edgecolors='k', s=30, label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ideal Fit')
plt.title('Actual vs Predicted Energy Consumption (Scatter)')
plt.xlabel('Actual Energy (Wh)')
plt.ylabel('Predicted Energy (Wh)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot 3: Time Series / Line Plot Comparison
plt.figure(figsize=(10, 5))
plt.plot(y_test[:100], label='Actual Values', color='#2b5c8f', linewidth=1.5)
plt.plot(y_pred[:100], label='Predicted Values', color='#d9534f', linestyle='--', linewidth=1.5)
plt.xlabel('Sample Index')
plt.ylabel('Energy Consumption (Wh)')
plt.title('Actual vs Predicted Energy Consumption (First 100 Samples)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()